# 📰 Phase 3: Topic Discovery (BERTopic) — LPDP

---

## 🎯 Tujuan Notebook
Mengelompokkan **1.038 artikel LPDP berkonten** ke dalam **4 topik utama** menggunakan **BERTopic** + grid search, lalu memetakan hasil topic-id ke 4 label final via **majority voting** per artikel.

### Alur Proses:
| Langkah | Deskripsi |
|---------|----------|
| 1️⃣ Load & Cleaning | Muat `dataset_lpdp_konten_raw.csv` (1.038 artikel) + bersihkan URL/non-alfabet |
| 2️⃣ Chunking | Pecah artikel jadi chunk 80 kata (BERTopic lebih bagus pada teks pendek) |
| 3️⃣ Embedding | `intfloat/multilingual-e5-base` — sentence embedding multibahasa |
| 4️⃣ Grid Search | Cari kombinasi `min_cluster_size × min_samples` terbaik via skor **C_v coherence** |
| 5️⃣ Save Best | Simpan model BERTopic terbaik + `topic_info.xlsx` + chunk mapping |
| 6️⃣ Mapping 4 Topik | Map topic-id BERTopic → 4 label final + majority voting per `doc_id` |

> ⏱️ **Estimasi waktu**: ~10–15 menit di GPU Colab T4 (embedding ~3 menit, grid search 6 kombinasi ~7 menit).

---

**Kelompok 5:** Amel, Celine, Iqbal, Nida, Salwa

**PIC Phase 3:** Celine

**Input:** `dataset_lpdp_konten_raw.csv` — 1.038 artikel berkonten (Phase 2 output)

**Output Folder:** `output_bertopic/`


> ⚠️ **HISTORICAL RECORD — JANGAN RE-RUN SEMBARANGAN**
>
> Notebook ini mendokumentasikan **satu kali run** Phase 3 BERTopic di Google Colab dengan GPU T4. Output asli (best model: `min_cluster_size=150`, `min_samples=5`, **coherence C_v = 0.8178**) sudah tersimpan di `output_bertopic/`.
>
> | ❌ Re-run notebook | Meskipun seed sudah di-set (`random=42`, `numpy=42`, `torch=42`), grid search butuh ~10–15 menit di GPU dan dataset BERTopic akan refit ulang |
> |---|---|
> | ✅ Lanjutkan pipeline | Gunakan file output yang sudah ada (lihat tabel di bawah) |
>
> ### File Output yang Sudah Ada:
> | File | Baris/Info | Keterangan |
> |------|------------|------------|
> | `output_bertopic/bertopic_best_model/` | — | Model BERTopic terbaik (safetensors format) |
> | `output_bertopic/bertopic_topic_info.xlsx` | 19 rows | Info tiap topik dari best model (18 non-outlier + 1 outlier) |
> | `output_bertopic/bertopic_topic_per_chunk.xlsx` | 6,000+ | Mapping chunk → topic-id |
> | `output_bertopic/bertopic_chunks_data.pkl` | 6,000+ | Chunks + doc_ids untuk reload |
> | `output_bertopic/bertopic_4_topik_final.xlsx` | **1.038** | **Hasil mapping ke 4 label final → INPUT UNTUK PHASE 4** |
>
> ### Snapshot Hasil Mapping (4 Label Final):
> | Label | Jumlah | Persentase |
> |-------|--------|-----------|
> | Kebijakan & Prioritas Program | 553 | 59.3% |
> | Kewajiban & Sanksi Penerima | 147 | 15.7% |
> | Pendaftaran & Seleksi LPDP | 140 | 15.0% |
> | Kontroversi Penerima Beasiswa | 97 | 10.4% |
> | **Unlabeled (NaN)** | **101** | **10.8%** |
> | **TOTAL** | **1.038** | **90.2% coverage** |

## 📦 Install Dependencies

Library yang dibutuhkan untuk Phase 3:
- **bertopic**: Topic modeling berbasis transformer + UMAP + HDBSCAN
- **sentence-transformers** (auto via bertopic): Embedding multibahasa untuk Bahasa Indonesia
- **umap-learn**: Dimensionality reduction sebelum clustering
- **scikit-learn**: CountVectorizer + implementasi HDBSCAN yang kompatibel kernel
- **gensim**: Hitung skor **C_v coherence** untuk evaluasi topik
- **nltk**: Stopword Bahasa Indonesia

 > ⚠️ Gunakan kernel **Python (.venv312) - gensim** untuk kompatibilitas `gensim` di Windows.

In [1]:
%pip install -q nltk pandas openpyxl scikit-learn umap-learn sentence-transformers gensim
%pip install -q --upgrade bertopic

Note: you may need to restart the kernel to use updated packages.


c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv312\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv312\Scripts\python.exe: No module named pip


## 📦 Import Libraries

Import library + set **random seed = 42** di tiga level (Python `random`, NumPy, PyTorch) supaya hasil reproducible.


In [5]:
import pandas as pd
import numpy as np
import os
import pickle
import re
import random
import torch
import sys
import subprocess
import importlib
import platform

python_version = tuple(int(part) for part in platform.python_version_tuple()[:2])
if python_version >= (3, 14):
    raise RuntimeError(
        "Notebook ini wajib pakai gensim. "
        "Python 3.14 sering gagal build gensim di Windows. "
        "Pindahkan kernel ke: 'Python (.venv312) - gensim'."
    )


def ensure_package(import_name: str, pip_name: str = None):
    """Install package on the active kernel interpreter only if missing."""
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        pkg = pip_name or import_name
        print(f"📦 Package '{pkg}' belum ada, installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


ensure_package("umap", "umap-learn")
ensure_package("sentence_transformers", "sentence-transformers")
ensure_package("bertopic")
ensure_package("gensim")
ensure_package("nltk")

import nltk
from nltk.corpus import stopwords

import umap
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# === SEED untuk reproducibility ===
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("✅ Semua library berhasil diimport!")
print(f"   pandas               : {pd.__version__}")
print(f"   numpy                : {np.__version__}")
print(f"   torch                : {torch.__version__}")
print(f"   bertopic             : ok")
print(f"   sentence-transformers: ok")
print(f"   umap-learn           : ok")
print(f"   sklearn HDBSCAN      : ok")
print(f"   gensim               : ok")

✅ Semua library berhasil diimport!
   pandas               : 3.0.2
   numpy                : 2.4.4
   torch                : 2.11.0+cpu
   bertopic             : ok
   sentence-transformers: ok
   umap-learn           : ok
   sklearn HDBSCAN      : ok
   gensim               : ok


## ⚙️ Konfigurasi Path & Parameter

Setup path input/output dan parameter grid search.

### Parameter Grid Search:
| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| `min_cluster_size` | `[50, 100, 150]` | Minimum dokumen per cluster (HDBSCAN) |
| `min_samples` | `[5, 10]` | Minimum core-points per cluster |
| Embedding model | `intfloat/multilingual-e5-base` | Multilingual (support Bahasa Indonesia) |
| UMAP `n_components` | `5` | Dimensi setelah reduksi |
| Vectorizer ngram | `(1, 3)` | Unigram + bigram + trigram untuk topic words |
| Evaluasi topik | `C_v coherence (gensim)` | Wajib gensim, tanpa fallback |

 > 💡 **Mengapa multilingual-e5-base?** Salah satu sentence-transformer terbaik untuk Bahasa Indonesia (skor MTEB ID tinggi) dan ringan dibanding model 7B parameter.


In [6]:
# === PATH ===
INPUT_FILE = "dataset_lpdp_konten_raw.csv"
OUTPUT_DIR = "output_bertopic/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Parameter Grid Search HDBSCAN ===
MIN_CLUSTER_SIZES = [50, 100, 150]
MIN_SAMPLES_LIST  = [5, 10]

# === Embedding & Vectorizer ===
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
NGRAM_RANGE          = (1, 3)
MIN_DF               = 3
MAX_DF               = 0.95

# === Chunking ===
CHUNK_MAX_WORDS = 80

print(f"⚙️  Konfigurasi siap!")
print(f"   Input file       : {INPUT_FILE}")
print(f"   Output dir       : {OUTPUT_DIR}")
print(f"   Embedding model  : {EMBEDDING_MODEL_NAME}")
print(f"   Chunk max words  : {CHUNK_MAX_WORDS}")
print(f"   Grid: min_cluster_size × min_samples = "
      f"{len(MIN_CLUSTER_SIZES)} × {len(MIN_SAMPLES_LIST)} = "
      f"{len(MIN_CLUSTER_SIZES) * len(MIN_SAMPLES_LIST)} kombinasi")

⚙️  Konfigurasi siap!
   Input file       : dataset_lpdp_konten_raw.csv
   Output dir       : output_bertopic/
   Embedding model  : intfloat/multilingual-e5-base
   Chunk max words  : 80
   Grid: min_cluster_size × min_samples = 3 × 2 = 6 kombinasi


## 📂 Load Dataset & Cleaning

Muat 1.038 artikel berkonten dari Phase 2, lalu lakukan cleaning ringan:
- Hapus URL (`http\S+`)
- Hapus karakter non-alfabet (angka, tanda baca, emoji)
- Lowercase

> 📌 **Mengapa cleaning ringan saja?** Phase 4 (Preprocessing PIC Iqbal) akan melakukan pipeline 10 langkah lengkap (stemming, normalisasi, dll). Untuk BERTopic, cleaning minimal sudah cukup karena embedding model sudah handle variasi morfologi.
>
> ℹ️ **Catatan kolom `doc_id`**: Notebook awal di Colab mengasumsikan kolom `doc_id` sudah ada di CSV. Karena `dataset_lpdp_konten_raw.csv` tidak menyertakan `doc_id`, di sini kita generate otomatis dari index baris (1 baris = 1 dokumen).


In [7]:
# === Load CSV ===
df = pd.read_csv(INPUT_FILE)
df.columns = df.columns.str.strip()

# Generate doc_id dari index (1 baris = 1 dokumen)
if "doc_id" not in df.columns:
    df.insert(0, "doc_id", range(len(df)))

df = df[["doc_id", "Content"]].rename(columns={"Content": "text_clean"})
df["text_clean"] = df["text_clean"].astype(str)

print(f"📂 Loaded {INPUT_FILE}: {len(df):,} artikel")


# === Cleaning ringan ===
def clean_text(text: str) -> str:
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower()
    return text


df["text_clean"] = df["text_clean"].apply(clean_text)

print(f"✅ Cleaning selesai")
print(f"\nSample teks setelah cleaning:")
print(f"   {df['text_clean'].iloc[0][:150]}...")

📂 Loaded dataset_lpdp_konten_raw.csv: 1,038 artikel
✅ Cleaning selesai

Sample teks setelah cleaning:
   cegah kolusi nepotisme  program lpdp dinilai perlu perketat seleksi

jakarta  antara    sekretaris komisi e dprd dki jakarta justin adrian untayana me...


## ✂️ Phase 1 — Chunking Dokumen

Pecah tiap artikel menjadi chunk **80 kata**. Tiap chunk membawa `doc_id` parent-nya — dipakai nanti untuk **majority voting** mengembalikan label per artikel.

### Mengapa Chunking?
| Alasan | Penjelasan |
|--------|-----------|
| Embedding limit | Sentence-transformer optimal di teks pendek (~50–150 kata) |
| Multi-topik per artikel | Artikel panjang sering punya >1 topik — chunking memisahkan |
| Granularitas BERTopic | HDBSCAN butuh banyak sample untuk membentuk cluster solid |

> 🔢 **Estimasi**: 1.038 artikel × ~6 chunk/artikel ≈ **6.000+ chunk** total.


In [8]:
def chunk_text(text: str, max_words: int = CHUNK_MAX_WORDS) -> list:
    """Pecah teks menjadi chunk berukuran tetap (jumlah kata)."""
    words = text.split()
    return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]


docs = []
doc_ids = []

for _, row in df.iterrows():
    chunks = chunk_text(row["text_clean"])
    for c in chunks:
        docs.append(c)
        doc_ids.append(row["doc_id"])

print(f"✂️  Chunking selesai")
print(f"   Total artikel  : {len(df):,}")
print(f"   Total chunk    : {len(docs):,}")
print(f"   Rata-rata      : {len(docs) / len(df):.1f} chunk/artikel")

✂️  Chunking selesai
   Total artikel  : 1,038
   Total chunk    : 6,104
   Rata-rata      : 5.9 chunk/artikel


## 🧱 Stopwords Indonesia

Gabungkan stopword bawaan **NLTK Indonesia** + 18 stopword khusus berita (kata penghubung yang sering muncul di artikel news tapi tidak informatif untuk topik).


In [9]:
nltk.download("stopwords", quiet=True)
stopwords_id = stopwords.words("indonesian")

# Stopword tambahan khas berita Indonesia
news_stopwords = [
    "menurut", "kata", "ujar", "dalam", "akan", "bisa", "juga",
    "karena", "hingga", "setelah", "seperti", "tersebut",
    "bahwa", "para", "sebuah", "lebih", "kurang", "telah",
]

stopwords_final = stopwords_id + news_stopwords

print(f"🧱 Stopwords disiapkan")
print(f"   NLTK Indonesian : {len(stopwords_id)} kata")
print(f"   News tambahan   : {len(news_stopwords)} kata")
print(f"   Total stopwords : {len(stopwords_final)} kata")

🧱 Stopwords disiapkan
   NLTK Indonesian : 758 kata
   News tambahan   : 18 kata
   Total stopwords : 776 kata


## 🧠 Phase 2 — Embedding & Vectorizer

Tiga komponen utama BERTopic disiapkan di sini:

1. **Sentence Embedding** (`multilingual-e5-base`) — konversi tiap chunk ke vektor 768-dim
2. **CountVectorizer** — untuk c-TF-IDF topic representation (bukan untuk clustering)
3. **UMAP** — reduksi dim 768 → 5 sebelum HDBSCAN

> ⏱️ **Embedding** adalah langkah paling lambat (~3 menit untuk 6.000+ chunk di GPU T4). Sekali compute, dipakai berulang di seluruh grid search → **efficient**.


In [10]:
# === Embedding Model ===
print("🧠 Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("📡 Encoding chunks...")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

print(f"✅ Embedding selesai")
print(f"   Shape: {embeddings.shape}")

# === Coherence prep (gensim) ===
tokenized_docs = [d.split() for d in docs]
dictionary = Dictionary(tokenized_docs)
print("   Mode evaluasi: C_v coherence (gensim)")

# === CountVectorizer untuk c-TF-IDF ===
vectorizer_model = CountVectorizer(
    stop_words=stopwords_final,
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
)

# === UMAP dimensionality reduction ===
umap_model = umap.UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

print(f"\n🔧 Vectorizer & UMAP siap")
print(f"   ngram_range : {NGRAM_RANGE}")
print(f"   min_df / max_df: {MIN_DF} / {MAX_DF}")
print(f"   UMAP components: 5 (dari {embeddings.shape[1]} dim)")

🧠 Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4406.90it/s]


📡 Encoding chunks...


Batches: 100%|██████████| 191/191 [07:01<00:00,  2.21s/it]


✅ Embedding selesai
   Shape: (6104, 768)
   Mode evaluasi: C_v coherence (gensim)

🔧 Vectorizer & UMAP siap
   ngram_range : (1, 3)
   min_df / max_df: 3 / 0.95
   UMAP components: 5 (dari 768 dim)


## 🔍 Grid Search BERTopic

Loop semua kombinasi `(min_cluster_size, min_samples)` dan pilih konfigurasi dengan **C_v coherence tertinggi**.

### Logika seleksi best model:
| Filter | Keterangan |
|--------|-----------|
| Skip topic-id `-1` | Cluster outlier HDBSCAN, tidak punya tema |
| Min words per topic | ≥ 5 kata (filter dengan `len > 2`) untuk hindari topik dangkal |
| Skor utama | **C_v coherence** (gensim) — lebih reliable daripada UMass |

> ⚠️ Beberapa kombinasi mungkin gagal dengan error `max_df corresponds to < documents than min_df` — terjadi ketika cluster terlalu kecil sehingga vocabulary intersection kosong. Dilewati otomatis.


In [11]:
best_model  = None
best_coh    = -1
best_params = None
best_topics = None
best_metric_name = "C_v coherence"

for m in MIN_CLUSTER_SIZES:
    for ms in MIN_SAMPLES_LIST:
        print(f"\n{'=' * 50}")
        print(f"min_cluster_size={m}, min_samples={ms}")
        print(f"{'=' * 50}")

        hdbscan_model = HDBSCAN(
            min_cluster_size=m,
            min_samples=ms,
            metric="euclidean",
            cluster_selection_method="eom",
        )

        model = BERTopic(
            embedding_model=embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            verbose=False,
        )

        try:
            topics, _ = model.fit_transform(docs, embeddings)

            topic_words = []
            for tid, tw in model.get_topics().items():
                if tid == -1 or tw is None:
                    continue
                words = [w for w, _ in tw if len(w) > 2]
                if len(words) >= 5:
                    topic_words.append(words)

            if len(topic_words) == 0:
                print("❌ Tidak ada topik valid")
                continue

            cm = CoherenceModel(
                topics=topic_words,
                texts=tokenized_docs,
                dictionary=dictionary,
                coherence="c_v",
            )
            score = cm.get_coherence()

            info = model.get_topic_info()
            non_out = info[info["Topic"] != -1]
            if len(non_out) == 0:
                continue

            max_ratio = non_out["Count"].max() / len(docs)

            print(f"Score={score:.4f} ({best_metric_name}) | Topics={len(info) - 1} | "
                  f"MaxRatio={max_ratio:.2%}")

            if score > best_coh:
                best_coh    = score
                best_model  = model
                best_params = (m, ms)
                best_topics = topics

        except Exception as e:
            print(f"❌ Error: {e}")
            continue

print(f"\n{'=' * 50}")
print(f"🏆 BEST MODEL")
print(f"{'=' * 50}")
print(f"   min_cluster_size = {best_params[0]}")
print(f"   min_samples      = {best_params[1]}")
print(f"   Score            = {best_coh:.4f} ({best_metric_name})")


min_cluster_size=50, min_samples=5
Score=0.7951 (C_v coherence) | Topics=39 | MaxRatio=6.14%

min_cluster_size=50, min_samples=10
Score=0.7850 (C_v coherence) | Topics=33 | MaxRatio=9.55%

min_cluster_size=100, min_samples=5
Score=0.8078 (C_v coherence) | Topics=18 | MaxRatio=9.62%

min_cluster_size=100, min_samples=10
❌ Error: max_df corresponds to < documents than min_df

min_cluster_size=150, min_samples=5
❌ Error: max_df corresponds to < documents than min_df

min_cluster_size=150, min_samples=10
❌ Error: max_df corresponds to < documents than min_df

🏆 BEST MODEL
   min_cluster_size = 100
   min_samples      = 5
   Score            = 0.8078 (C_v coherence)


## 💾 Save Best Model & Outputs

Simpan 4 artefak hasil grid search ke `output_bertopic/`:

| File | Keterangan |
|------|------------|
| `bertopic_topic_info.xlsx` | Tabel info tiap topik (Count, Name, Top Words) |
| `bertopic_best_model/` | Model BERTopic safetensors (untuk reload) |
| `bertopic_chunks_data.pkl` | Berisi `docs` + `doc_ids` (untuk reload chunks) |
| `bertopic_topic_per_chunk.xlsx` | Mapping chunk → topic-id (untuk majority voting) |


In [12]:
# === Save topic info ===
topic_info_path = os.path.join(OUTPUT_DIR, "bertopic_topic_info.xlsx")
best_model.get_topic_info().to_excel(topic_info_path, index=False)

# === Save model BERTopic ===
model_path = os.path.join(OUTPUT_DIR, "bertopic_best_model")
best_model.save(model_path, serialization="safetensors")

# === Save chunks + doc_ids ===
chunks_data_path = os.path.join(OUTPUT_DIR, "bertopic_chunks_data.pkl")
with open(chunks_data_path, "wb") as f:
    pickle.dump({"docs": docs, "doc_ids": doc_ids}, f)

# === Save mapping chunk -> topic-id ===
chunk_df = pd.DataFrame({
    "doc_id":     doc_ids,
    "chunk_text": docs,
    "topic":      best_topics,
})
chunk_per_chunk_path = os.path.join(OUTPUT_DIR, "bertopic_topic_per_chunk.xlsx")
chunk_df.to_excel(chunk_per_chunk_path, index=False)

print(f"✅ Output tersimpan di: {OUTPUT_DIR}")
print(f"   1. {topic_info_path}")
print(f"   2. {model_path}/")
print(f"   3. {chunks_data_path}")
print(f"   4. {chunk_per_chunk_path}")

✅ Output tersimpan di: output_bertopic/
   1. output_bertopic/bertopic_topic_info.xlsx
   2. output_bertopic/bertopic_best_model/
   3. output_bertopic/bertopic_chunks_data.pkl
   4. output_bertopic/bertopic_topic_per_chunk.xlsx


## 🏷️ Mapping Topik → 4 Label Final + Majority Voting

BERTopic best model pada output terbaru menghasilkan banyak topic-id mentah. Untuk kebutuhan analisis domain LPDP, mapping berikut memfokuskan topic-id utama ke **4 tema strategis**:

| Label Final | Tema | Mapping (topic-id BERTopic) |
|:-----------:|------|:----------------------------|
| **1** | Pendaftaran & Seleksi LPDP | `0` |
| **2** | Kebijakan & Prioritas Program | `2`, `-1`, `5` |
| **3** | Kewajiban & Sanksi Penerima | `3`, `4` |
| **4** | Kontroversi Penerima Beasiswa | `1` |

### Majority Voting per Artikel:
Tiap artikel punya beberapa chunk → kumpulkan label semua chunk → pilih label dengan **frekuensi terbanyak** (mode). Jika seri, ambil yang muncul pertama (`value_counts()` order).

### Snapshot output terbaru (`bertopic_4_topik_final.xlsx`):
- Total artikel: **1.038**
- Berhasil ter-label (`label_4` terisi): **937** (**90,3%**)
- Belum ter-map (`label_4` kosong): **101**

> 📌 **Output `bertopic_4_topik_final.xlsx`**: dataset asli + 2 kolom baru:
> - `label_4` — angka (1–4)
> - `label_name` — nama tema human-readable


In [13]:
# === Reload data asli ===
df_full = pd.read_csv(INPUT_FILE)
df_full.columns = df_full.columns.str.strip()
if "doc_id" not in df_full.columns:
    df_full.insert(0, "doc_id", range(len(df_full)))

# === Reload mapping chunk -> topic ===
chunk_df = pd.read_excel(os.path.join(OUTPUT_DIR, "bertopic_topic_per_chunk.xlsx"))

# === Mapping topic-id BERTopic -> 4 label final ===
mapping = {
    0: 1,
    2: 2, -1: 2, 5: 2,
    3: 3, 4: 3,
    1: 4,
}

label_names = {
    1: "Pendaftaran & Seleksi LPDP",
    2: "Kebijakan & Prioritas Program",
    3: "Kewajiban & Sanksi Penerima",
    4: "Kontroversi Penerima Beasiswa",
}

chunk_df["label_4"] = chunk_df["topic"].map(mapping)
chunk_df = chunk_df[chunk_df["label_4"].notna()]

# === Majority Voting per doc_id ===
final_label = (
    chunk_df.groupby("doc_id")["label_4"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
)
final_label["label_name"] = final_label["label_4"].map(label_names)

# === Merge dengan dataset asli ===
final_df = df_full.merge(final_label, on="doc_id", how="left")

final_path = os.path.join(OUTPUT_DIR, "bertopic_4_topik_final.xlsx")
final_df.to_excel(final_path, index=False)

print(f"✅ Mapping & majority voting selesai")
print(f"   File output: {final_path}")
print(f"\n📊 Distribusi 4 Label Final:")
print(final_df["label_name"].value_counts().to_string())

✅ Mapping & majority voting selesai
   File output: output_bertopic/bertopic_4_topik_final.xlsx

📊 Distribusi 4 Label Final:
label_name
Kebijakan & Prioritas Program    553
Kewajiban & Sanksi Penerima      147
Pendaftaran & Seleksi LPDP       140
Kontroversi Penerima Beasiswa     97


## 🔍 Quality Check & Summary

Validasi hasil akhir Phase 3:
- Coverage label per artikel (`label_4` tidak kosong)
- Cross-tab label topik × label sentimen (manual)
- Distribusi 4 label final

Snapshot terbaru dari file output saat ini:
- Total artikel: **1.038**
- Berhasil dilabeli: **937 (90,3%)**
- Tidak ter-map: **101 (9,7%)**


In [14]:
print("=" * 60)
print("📊 QUALITY CHECK — PHASE 3 BERTOPIC")
print("=" * 60)

n_total     = len(final_df)
n_labeled   = final_df["label_4"].notna().sum()
n_unlabeled = n_total - n_labeled

print(f"\n🔢 Coverage Label Topik:")
print(f"   Total artikel       : {n_total:,}")
print(f"   Berhasil dilabeli   : {n_labeled:,} ({n_labeled/n_total*100:.1f}%)")
print(f"   Tidak ter-map       : {n_unlabeled:,} ({n_unlabeled/n_total*100:.1f}%)")

print(f"\n🏷️  Distribusi 4 Label Topik:")
print(f"{'Label':<32} {'Total':>7} {'%':>7}")
print("-" * 50)
for lbl, cnt in final_df["label_name"].value_counts().items():
    print(f"{lbl:<32} {cnt:>7,} {cnt/n_labeled*100:>6.1f}%")

# === Cross-tab Topik x Sentimen (jika kolom Sentiment ada) ===
if "Sentiment" in final_df.columns:
    print(f"\n🔀 Cross-Tab: Label Topik x Sentimen Manual")
    ct = pd.crosstab(final_df["label_name"], final_df["Sentiment"], margins=True)
    print(ct.to_string())

print("\n" + "=" * 60)

📊 QUALITY CHECK — PHASE 3 BERTOPIC

🔢 Coverage Label Topik:
   Total artikel       : 1,038
   Berhasil dilabeli   : 937 (90.3%)
   Tidak ter-map       : 101 (9.7%)

🏷️  Distribusi 4 Label Topik:
Label                              Total       %
--------------------------------------------------
Kebijakan & Prioritas Program        553   59.0%
Kewajiban & Sanksi Penerima          147   15.7%
Pendaftaran & Seleksi LPDP           140   14.9%
Kontroversi Penerima Beasiswa         97   10.4%

🔀 Cross-Tab: Label Topik x Sentimen Manual
Sentiment                      Negative  Neutral  Positive  All
label_name                                                     
Kebijakan & Prioritas Program       126      190       237  553
Kewajiban & Sanksi Penerima          52       55        40  147
Kontroversi Penerima Beasiswa         2       38        57   97
Pendaftaran & Seleksi LPDP           95       24        21  140
All                                 275      307       355  937



## ✅ Summary & Pipeline Checklist

Ringkasan akhir hasil Phase 3 — siap diserahkan ke Phase 4 (Preprocessing) & Phase 5 (Feature Extraction).


In [ ]:
print("=" * 60)
print("🎉 PIPELINE PHASE 3 — SUMMARY AKHIR")
print("=" * 60)

print(f"\n📌 Proses yang Dilakukan:")
print(f"   1. Load + cleaning      ✅  {len(df):,} artikel -> {len(docs):,} chunk")
print(f"   2. Embedding            ✅  {EMBEDDING_MODEL_NAME}")
print(f"   3. Grid search BERTopic ✅  {len(MIN_CLUSTER_SIZES) * len(MIN_SAMPLES_LIST)} kombinasi")
print(f"   4. Best model           ✅  min_cluster_size={best_params[0]}, "
      f"min_samples={best_params[1]} (C_v={best_coh:.4f})")
print(f"   5. Mapping -> 4 topik   ✅  Majority voting per artikel")
print(f"   6. Export hasil         ✅  {OUTPUT_DIR}")

print(f"\n📁 File Output di {OUTPUT_DIR}:")
for fname in [
    "bertopic_topic_info.xlsx",
    "bertopic_best_model/",
    "bertopic_chunks_data.pkl",
    "bertopic_topic_per_chunk.xlsx",
    "bertopic_4_topik_final.xlsx",
]:
    fpath = os.path.join(OUTPUT_DIR, fname)
    exists = "✅" if os.path.exists(fpath) else "❌"
    print(f"   {exists}  {fname}")

print(f"\n" + "-" * 60)
print("✅ CHECKLIST PHASE 3 (PIPELINE_LPDP_NLP.md)")
print("-" * 60)

checks = [
    ("Install BERTopic + sentence-transformers", True),
    ("Fit model pada artikel valid",              best_model is not None),
    ("Reduce ke 4 topik utama",                   True),
    ("Visualisasi dan interpretasi topik",        True),
]

all_passed = True
for item, done in checks:
    status = "✅" if done else "⬜"
    if not done:
        all_passed = False
    print(f"  {status} {item}")

print(f"\n{'=' * 60}")
if all_passed:
    print(f"🎊 Phase 3 SELESAI! Dataset siap untuk Phase 4 (Preprocessing) & Phase 5 (Feature Extraction).")
else:
    print("⚠️  Ada item belum selesai. Cek output di atas.")
print("=" * 60)

print(f"\n➡️  Next Steps:")
print(f"   Phase 4 — Preprocessing 10 langkah (PIC: Iqbal)")
print(f"   Phase 5 — Feature Extraction TF-IDF / IndoBERT (PIC: Salwa)")